In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import scipy.io as sio
from dataclasses import dataclass
from typing import List, Tuple
import os
from dotenv import load_dotenv
load_dotenv()
import tidy3d as td
from tidy3d import web
import numpy as np
from pathlib import Path
from stl import mesh
import matplotlib.pyplot as plt
import re

In [2]:
import sys
import os

# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(fr'../../../../tidy3d'))

from AutomationModule import * 

import AutomationModule as AM

In [3]:
tidy3dAPI = os.environ["API_TIDY3D_KEY"]


In [4]:
lambdas = np.array([8,2])
sphere,phi,theta= get_sphere(
    1000,
    theta_range=(0.0, np.pi/2),
    degrees=False
)

In [5]:
folder_path = rf"./structures/n_3.40/Schulz/AR Hole 2.5/ff_0p2172/ffh_0.225_dist"
project_name = "20260423 LSU Transmission n_3.4 ff_0.2172 ffh_0.225 Schulz Diffraction No FixedAngleSpec"
postprocess_results = []
runtime_ps = 25e-12
min_steps_per_lambda = 20
h5_bg = None
ref = True
cuts=np.array([2,3,4,5,6,7,8,10,11,12,14,15])/11.44
# cuts=np.array([9])/11.44
for direction in ["z"]: 
    for dirpath, dirnames, filenames in os.walk(folder_path):
        print(f"Processing directory: {dirpath}")
        try:
            for filename in filenames:
                print(filename)
                if filename.endswith(".h5"):
                    n_value = float(re.search(r'n_([+-]?\d+(?:\.\d+)?)', filename).group(1))
                    ff = float(re.search(r'fforiginal_([+-]?\d+(?:\.\d+)?)', filename).group(1))
                    z = float(re.search(r'z_([+-]?\d+(?:\.\d+)?)', filename).group(1))
                    index = re.search(r'(\d+(?:\.\d+)?)\.h5$', filename).group(1)
                    # if float(index) > 0:
                    #     continue #We're only running one sample
                    for cut in cuts:
                        if not (Path(filename).suffix==".h5" or Path(filename).suffix==".stl"):
                            continue 
                        if os.path.isfile(os.path.join(dirpath, filename)):
                            file=os.path.join(dirpath, filename)
                            structure_1 = AM.loadAndRunStructure(key = tidy3dAPI, file_path=file
                                                            ,direction=direction, lambda_range=lambdas,
                                                            box_size=11.44,runtime_ps=runtime_ps,min_steps_per_lambda=min_steps_per_lambda,
                                                           scaling=1,shuoff_condtion=1e-20, verbose=True, 
                                                           monitors=["flux","diffraction"], flux_monitor_position=18,cell_size_manual=44,
                                                           freqs=400, 
                                                           cut_condition=cut, source="planewave", absorbers=130, use_permittivity=False,sim_name=rf"{Path(filename).stem}_size_{cut}" + (rf"_bg_{h5_bg}" if h5_bg else ""),h5_bg=h5_bg,
                                                           )
                            folder_desc = rf"../../../data/{project_name}/n_{n_value:.2f}/z_{z}/size_{cut*11.44:.2f}"
                            os.makedirs(folder_desc, exist_ok=True)
                            sim_name=rf"LSU_ffr_{ff}_size_{cut}_n_{n_value:.2f}_z_{z}_sample_{index}"
                           
                            if os.path.exists(os.path.join(folder_desc, sim_name+".txt")):
                                print("Exist!")
                            else:
                                if ref:
                                 sim0=(structure_1.sim).copy(update={"structures":[]})
                                 id0 =web.upload(sim0, folder_name=project_name,task_name=sim_name+"_0", verbose=True)
                                 web.start(task_id = id0)
                                 web.monitor(task_id=id0,verbose=True)
                                 with open(os.path.join(rf"../../../data/{project_name}", "reference.txt"), "w") as file:
                                    # Write the string to the file
                                    file.write('\n'+id0)
                                 ref=False

                                id =web.upload(structure_1.sim, folder_name=project_name,task_name=sim_name, verbose=True)
                                structure_1.plot_sim_layout()
                                ids = '\n' + id
                                with open(os.path.join(folder_desc, sim_name+".txt"), "w") as file:
                                    # Write the string to the file
                                    file.write(ids)
                                web.start(task_id = id)
                                web.monitor(task_id=id,verbose=True)
                            del structure_1

        except Exception as e:
            print(f"Error processing {dirpath}: {e}")
            continue


Processing directory: ./structures/n_3.40/Schulz/AR Hole 2.5/ff_0p2172/ffh_0.225_dist
Processing directory: ./structures/n_3.40/Schulz/AR Hole 2.5/ff_0p2172/ffh_0.225_dist\z_100
n_3.40_ffh_0.2204_ffr_0.1693_fforiginal_0.2172_z_100_2.h5
Configured successfully.
Exist!
Configured successfully.
Exist!
Configured successfully.
Exist!
Configured successfully.
Exist!
Configured successfully.
Exist!
Configured successfully.
Exist!
Error processing ./structures/n_3.40/Schulz/AR Hole 2.5/ff_0p2172/ffh_0.225_dist\z_100: HTTPSConnectionPool(host='tidy3d-api.simulation.cloud', port=443): Max retries exceeded with url: /apikey (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000002890E050FB0>: Failed to resolve 'tidy3d-api.simulation.cloud' ([Errno 11001] getaddrinfo failed)"))
Processing directory: ./structures/n_3.40/Schulz/AR Hole 2.5/ff_0p2172/ffh_0.225_dist\z_5
n_3.40_ffh_0.2187_ffr_0.1697_fforiginal_0.2172_z_5_3.h5
Configured successfully.
Exist!
Configured succ